# 自建长期记忆全流程

| 步骤 | 学什么 | 看哪段源码 / 调用 |
|------|--------|-------------------|
| 源码 A | BM25 + 记忆抽取/合并 | `HybridRetriever`、`MemoryCompressor` |
| 源码 B | 存进 Chroma、检索、拼 Prompt | `SelfBuiltLongTermMemoryImproved` |
|模块 1～5 | 动手跑一遍 | 调用上面的类 |
| 模块 | 本课做什么 | 对应代码 |
| **1. 抽取** | LLM 把自然语言压成 `summary / key_points / importance` | `MemoryCompressor.extract_memory` |
| **2. 合并** | 两条相似记忆合成一条 | `MemoryCompressor.merge_memories` |
| **3. 存储** | 写入管道：去重 → 合并 → Chroma 索引 | `ingest_long_term` / `MemoryWritePipeline` |
| **4. 检索** | 向量 + jieba BM25 + RRF×向量相似度 + 时间/重要性乘性重排 | `search_memories` |
| **5. 使用** | 短期 Checkpoint 模拟 + 长期记忆拼进 Prompt（可选调 LLM） | `build_prompt` + `ChatOpenAI` |

向量默认走 **阿里云 DashScope**（`.env` 里配 `DASHSCOPE_API_KEY`）。


## 0. 环境

```bash
pip install langchain-core langchain-openai chromadb rank-bm25 jieba openai python-dotenv httpx numpy ipywidgets
```

在项目根目录放 `.env`（只写 `KEY=VALUE`，不要写 YAML 块）。


In [16]:
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd()
ROOT = Path(r"d:\myproject\enterprise_bench_langchain")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if sys.platform == "win32":
    for _s in (sys.stdout, sys.stderr):
        try:
            _s.reconfigure(encoding="utf-8")
        except Exception:
            pass

load_dotenv(ROOT / ".env")

print("项目根目录:", ROOT)
print("DASHSCOPE_API_KEY:", "已配置" if os.getenv("DASHSCOPE_API_KEY", "").strip() else "未配置")
print("LLM API:", "已配置" if (os.getenv("OPENAI_API_KEY") or os.getenv("OPENROUTER_API_KEY") or "").strip() else "未配置")

项目根目录: d:\myproject\enterprise_bench_langchain
DASHSCOPE_API_KEY: 已配置
LLM API: 已配置


## 架构（先建立直觉）

```
用户说的话 ──►【抽取】──►【去重/合并】──► 变成向量 ──► 存进 Chroma
新问题     ──►【向量检索 + BM25】──►【重排】──► 和最近聊天一起 ──► 拼成 Prompt ──► 问大模型
```


## 源码 A：混合检索 + 记忆抽取/合并

**请展开本格阅读**，运行后定义 `HybridRetriever`（jieba BM25 + RRF）、`MemoryCompressor`。


In [17]:
# ========== 源码 A：混合检索 + 记忆抽取/合并（来自 EmbeddingFactory.py）==========
import json
import re
import uuid
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
from rank_bm25 import BM25Okapi

try:
    import jieba

    def _bm25_tokenize(text: str) -> List[str]:
        """中文 jieba 分词；英文仍按空格切分。"""
        return [t.strip() for t in jieba.cut(text.lower()) if t.strip()]
except ImportError:
    def _bm25_tokenize(text: str) -> List[str]:
        return text.lower().split()


class HybridRetriever:
    """混合检索器：jieba BM25 稀疏检索 + 向量排名 + RRF 融合（返回纯 RRF 分，不含向量幅度）"""

    def __init__(self, k1: float = 1.5, b: float = 0.75, rrf_k: int = 60):
        self.bm25 = None
        self.documents = []
        self.k1 = k1
        self.b = b
        self.rrf_k = rrf_k

    def fit(self, documents: List[str]):
        """用文档列表构建 BM25 索引（jieba 分词后统计词频/IDF）"""
        self.documents = documents
        tokenized_docs = [_bm25_tokenize(doc) for doc in documents]
        self.bm25 = BM25Okapi(tokenized_docs, k1=self.k1, b=self.b)

    def search(
        self, query: str, dense_scores: List[Tuple[int, float]], top_k: int = 5
    ) -> List[Tuple[int, float]]:
        """
        执行混合检索，返回 [(文档索引, rrf_score)]。
        dense_scores 中的相似度仅用于确定向量侧排名；幅度在 search_memories 中再乘回。
        """
        if not self.bm25 or not self.documents:
            # 无 BM25 时：用向量排名近似 RRF
            return [(idx, 1.0 / (self.rrf_k + rank)) for rank, (idx, _) in enumerate(dense_scores[:top_k])]

        tokenized_query = _bm25_tokenize(query)
        bm25_scores = self.bm25.get_scores(tokenized_query)
        bm25_ranked = np.argsort(bm25_scores)[::-1]

        rrf_scores: Dict[int, float] = {}
        for rank, (idx, _) in enumerate(dense_scores):
            rrf_scores[idx] = rrf_scores.get(idx, 0.0) + 1.0 / (self.rrf_k + rank)
        for rank, idx in enumerate(bm25_ranked):
            if bm25_scores[idx] > 0:
                rrf_scores[idx] = rrf_scores.get(idx, 0.0) + 1.0 / (self.rrf_k + rank)

        sorted_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
        return sorted_results[:top_k]


# ==============================
# 3. 记忆抽取与合并器
# ==============================
def _llm_text(response: Any) -> str:
    """从 LangChain AIMessage 取出文本（兼容空 content / 多段 content）。"""
    content = getattr(response, "content", None)
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        parts: List[str] = []
        for block in content:
            if isinstance(block, str):
                parts.append(block)
            elif isinstance(block, dict) and block.get("type") == "text":
                parts.append(str(block.get("text", "")))
        return "".join(parts).strip()
    return str(content or "").strip()


def _parse_json_from_llm(text: str) -> Dict[str, Any]:
    """解析 LLM 返回的 JSON（支持 ```json 代码块与夹杂说明文字）。"""
    raw = (text or "").strip()
    if not raw:
        raise ValueError("LLM 返回为空，无法解析 JSON")

    candidates = [raw]
    fence = re.search(r"```(?:json)?\s*([\s\S]*?)```", raw, re.IGNORECASE)
    if fence:
        candidates.insert(0, fence.group(1).strip())
    brace = re.search(r"\{[\s\S]*\}", raw)
    if brace:
        candidates.append(brace.group(0))

    last_err: Optional[Exception] = None
    for piece in candidates:
        try:
            obj = json.loads(piece)
            if isinstance(obj, dict):
                return obj
        except json.JSONDecodeError as e:
            last_err = e
            continue
    raise ValueError(f"无法从 LLM 输出解析 JSON: {last_err}")


class MemoryCompressor:
    """记忆抽取与合并器，自动将相似记忆合并为核心要点"""

    def __init__(self, llm):
        self.llm = llm

    def _invoke_json(self, prompt: str, *, retries: int = 2) -> Dict[str, Any]:
        last_err: Optional[Exception] = None
        for attempt in range(retries):
            p = prompt if attempt == 0 else (
                prompt.rstrip()
                + "\n\n请只输出一个合法 JSON 对象，不要 markdown、不要解释。"
            )
            try:
                return _parse_json_from_llm(_llm_text(self.llm.invoke(p)))
            except Exception as e:
                last_err = e
        raise ValueError(str(last_err))

    def _fallback_extract(self, content: str, metadata: Dict[str, Any] | None) -> Dict[str, Any]:
        snippet = content.strip().replace("\n", " ")
        summary = snippet[:120] + ("…" if len(snippet) > 120 else "")
        return {
            "id": str(uuid.uuid4()),
            "content": content,
            "summary": summary,
            "key_points": [snippet] if snippet else [],
            "importance": float((metadata or {}).get("importance", 0.5)),
            "timestamp": datetime.now().isoformat(),
            "metadata": metadata or {},
        }

    def extract_memory(self, content: str, metadata: Dict[str, Any] = None) -> Dict[str, Any]:
        """从原始内容中抽取结构化记忆"""
        prompt = f"""请从以下内容中提取核心记忆点，保留所有重要信息：
        {content}
        
        只输出一个 JSON 对象，字段如下：
        {{
            "summary": "一句话总结",
            "key_points": ["要点1", "要点2"],
            "importance": 0.5
        }}
        importance 为 0.0 到 1.0 之间的数字。"""

        try:
            extracted = self._invoke_json(prompt)
        except Exception:
            return self._fallback_extract(content, metadata)

        imp = extracted.get("importance", 0.5)
        try:
            imp = float(imp)
        except (TypeError, ValueError):
            imp = 0.5

        return {
            "id": str(uuid.uuid4()),
            "content": content,
            "summary": str(extracted.get("summary", content)),
            "key_points": list(extracted.get("key_points") or []),
            "importance": imp,
            "timestamp": datetime.now().isoformat(),
            "metadata": metadata or {},
        }

    def merge_memories(self, memories: List[Dict[str, Any]]) -> Dict[str, Any]:
        """合并多个相似记忆为一个综合记忆"""
        memories_text = "\n\n".join([f"记忆{i + 1}:\n{mem['content']}" for i, mem in enumerate(memories)])

        prompt = f"""请将以下多个相似记忆合并为一个综合记忆，保留所有独特的重要信息：
        {memories_text}
        
        只输出一个 JSON 对象：
        {{
            "summary": "合并后的一句话总结",
            "key_points": ["合并后的要点1", "合并后的要点2"],
            "importance": 0.5
        }}"""

        try:
            merged = self._invoke_json(prompt)
        except Exception:
            base = memories[0]
            return {
                **base,
                "id": str(uuid.uuid4()),
                "metadata": {"merged_from": [mem["id"] for mem in memories], **(base.get("metadata") or {})},
                "is_merged": True,
            }

        imp = merged.get("importance", 0.5)
        try:
            imp = float(imp)
        except (TypeError, ValueError):
            imp = 0.5
        key_points = list(merged.get("key_points") or [])

        return {
            "id": str(uuid.uuid4()),
            "content": "\n".join(key_points) if key_points else memories_text,
            "summary": str(merged.get("summary", memories[0].get("summary", ""))),
            "key_points": key_points,
            "importance": imp,
            "timestamp": datetime.now().isoformat(),
            "metadata": {"merged_from": [mem["id"] for mem in memories]},
            "is_merged": True,
        }

print('✓ 源码 A 已执行：HybridRetriever、MemoryCompressor')

✓ 源码 A 已执行：HybridRetriever、MemoryCompressor


## 源码 B：长期记忆系统

**请展开本格阅读**，运行后定义 `SelfBuiltLongTermMemoryImproved` 等类。


In [18]:
# ========== 源码 B：长期记忆系统（来自 SelfBuiltLongTermMemory_improve.py）==========
from __future__ import annotations

import asyncio
import json
import os
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional

import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

load_dotenv(ROOT / ".env")



def default_llm():
    """读取 .env，创建对话用大模型（课堂统一入口）。"""
    from langchain_openai import ChatOpenAI
    import httpx

    kw: dict = {
        "model": os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        "temperature": 0,
        "api_key": (os.getenv("OPENAI_API_KEY") or os.getenv("OPENROUTER_API_KEY") or "").strip() or None,
    }
    base = (os.getenv("OPENAI_BASE_URL") or "").strip()
    if base:
        kw["base_url"] = base.rstrip("/")
    if os.getenv("OPENAI_SSL_VERIFY", "1").strip().lower() in ("0", "false", "no", "off"):
        kw["http_client"] = httpx.Client(verify=False)
    return ChatOpenAI(**kw)



_NOTEBOOK_STATE: dict = {}


def close_chroma_ltm(ltm_obj=None) -> None:
    """释放 Chroma 引用（尽量让 GC 回收文件句柄）。"""
    import gc
    import time

    target = ltm_obj if ltm_obj is not None else _NOTEBOOK_STATE.get("ltm")
    if target is not None:
        if hasattr(target, "close"):
            try:
                target.close()
            except Exception:
                pass
        try:
            target.collection = None
        except Exception:
            pass
        try:
            target.client = None
        except Exception:
            pass
    if ltm_obj is None:
        _NOTEBOOK_STATE.pop("ltm", None)
        _NOTEBOOK_STATE.pop("key", None)
    gc.collect()
    time.sleep(0.2)


def reset_chroma_directory(path: Path, ltm_obj=None) -> bool:
    """尝试删除整个 chroma 目录。Windows 上常会失败，返回 False 即可，不要抛错。"""
    import shutil
    import time

    close_chroma_ltm(ltm_obj)
    if not path.exists():
        return True
    for _ in range(6):
        try:
            shutil.rmtree(path)
            return True
        except PermissionError:
            time.sleep(0.5)
    print(
        f"⚠️ 未能删除文件夹 {path}（被系统占用）。"
        "将改为「复用 ltm + 只清空 collection」，不影响课堂演示。"
    )
    return False


def get_or_reset_ltm(
    *,
    user_id: str,
    persist_directory: str | Path,
    collection_name: str = "notebook_long_term_v1",
    force_rebuild_directory: bool = False,
    **kwargs: Any,
) -> "SelfBuiltLongTermMemoryImproved":
    """Notebook 专用：第二次运行优先复用 ltm 并 clear_all_memories，避免 WinError 32。"""
    import gc
    import time

    path = Path(persist_directory)
    path.mkdir(parents=True, exist_ok=True)
    key = (str(path.resolve()), collection_name, user_id)

    if force_rebuild_directory:
        reset_chroma_directory(path, _NOTEBOOK_STATE.get("ltm"))

    existing = _NOTEBOOK_STATE.get("ltm")
    if (
        isinstance(existing, SelfBuiltLongTermMemoryImproved)
        and _NOTEBOOK_STATE.get("key") == key
        and not force_rebuild_directory
    ):
        existing.clear_all_memories()
        print("♻️ 已复用 ltm，并清空 collection（未删磁盘目录）")
        globals()["ltm"] = existing
        return existing

    if isinstance(existing, SelfBuiltLongTermMemoryImproved):
        close_chroma_ltm(existing)
        gc.collect()
        time.sleep(0.3)

    ltm = SelfBuiltLongTermMemoryImproved(
        user_id=user_id,
        persist_directory=str(path),
        collection_name=collection_name,
        **kwargs,
    )
    _NOTEBOOK_STATE["ltm"] = ltm
    _NOTEBOOK_STATE["key"] = key
    globals()["ltm"] = ltm
    return ltm

MEMORY_PROMPT_TEMPLATE = """你是一个有记忆的智能助手。请结合【最近对话】与【相关长期记忆】回答用户问题。

【最近对话】（来自 Checkpoint / 本线程短期缓冲，按时间顺序）
{recent_dialogue}

【相关长期记忆】（向量检索 + 重排后的 top-k）
{memory_context}

【当前问题】
{query}
"""


# ---------------------------------------------------------------------------
# 短期：模拟 LangGraph Checkpoint 中的 messages（只追加、不向量检索）
# ---------------------------------------------------------------------------
class ThreadShortTermMemory:
    """thread_id 隔离的短期对话缓冲，等价于 Checkpoint 里本会话的 messages 子集。"""

    def __init__(self, max_turns: int = 20) -> None:
        self.max_turns = max_turns
        self._threads: Dict[str, List[Dict[str, str]]] = {}

    def append(self, thread_id: str, role: str, content: str) -> None:
        self._threads.setdefault(thread_id, []).append({"role": role, "content": content})
        # 只保留最近 max_turns 条
        if len(self._threads[thread_id]) > self.max_turns:
            self._threads[thread_id] = self._threads[thread_id][-self.max_turns :]

    def format_recent(self, thread_id: str, last_n: int = 6) -> str:
        turns = self._threads.get(thread_id, [])[-last_n:]
        if not turns:
            return "（无历史对话）"
        return "\n".join(f"{t['role']}: {t['content']}" for t in turns)


# ---------------------------------------------------------------------------
# 写入管道：抽取 → 去重 → 合并 → 索引
# ---------------------------------------------------------------------------
@dataclass
class WritePipelineConfig:
    dedupe_similarity_threshold: float = 0.82  # Chroma 距离转相似度后的合并阈值
    merge_on_duplicate: bool = True


class MemoryWritePipeline:
    def __init__(
        self,
        collection,
        compressor: MemoryCompressor,
        hybrid_retriever: HybridRetriever,
        *,
        user_id: str,
        config: WritePipelineConfig | None = None,
        on_index_refreshed=None,
    ) -> None:
        self.collection = collection
        self.compressor = compressor
        self.hybrid_retriever = hybrid_retriever
        self.user_id = user_id
        self.config = config or WritePipelineConfig()
        self._on_index_refreshed = on_index_refreshed
        self._queue: asyncio.Queue[Dict[str, Any]] = asyncio.Queue()
        self._worker_task: Optional[asyncio.Task] = None

    def _distance_to_similarity(self, distance: float) -> float:
        return max(0.0, min(1.0, 1.0 - float(distance) / 2.0))

    def _sync_index_state(self, ltm: "SelfBuiltLongTermMemoryImproved") -> None:
        all_data = self.collection.get()
        ltm.documents = all_data.get("documents") or []
        ltm.metadatas = all_data.get("metadatas") or []
        ltm.ids = all_data.get("ids") or []
        if ltm.documents:
            ltm.hybrid_retriever.fit(ltm.documents)
        if self._on_index_refreshed:
            self._on_index_refreshed()

    def ingest_sync(
        self,
        ltm: "SelfBuiltLongTermMemoryImproved",
        content: str,
        *,
        memory_type: str = "general",
        importance: float | None = None,
        extra_metadata: Dict[str, Any] | None = None,
    ) -> str:
        """同步执行完整写入管道（课堂演示用）。"""
        extracted = self.compressor.extract_memory(content, extra_metadata)
        imp = float(importance if importance is not None else extracted.get("importance", 0.5))

        # 去重：用摘要/原文做一次向量近邻
        dup_id: Optional[str] = None
        if self.collection.count() > 0:
            hits = self.collection.query(
                query_texts=[extracted["summary"]],
                n_results=3,
                where={"user_id": self.user_id},
            )
            if hits["ids"] and hits["ids"][0]:
                best_dist = hits["distances"][0][0]
                if self._distance_to_similarity(best_dist) >= self.config.dedupe_similarity_threshold:
                    dup_id = hits["ids"][0][0]

        if dup_id and self.config.merge_on_duplicate:
            existing = self.collection.get(ids=[dup_id])
            old_meta = existing["metadatas"][0]
            old_mem = {
                "id": dup_id,
                "content": existing["documents"][0],
                "summary": old_meta.get("summary", ""),
                "key_points": json.loads(old_meta.get("key_points", "[]")),
                "importance": old_meta.get("importance", 0.5),
            }
            merged = self.compressor.merge_memories([old_mem, extracted])
            memory_id = dup_id
            extra_metadata = {**(extra_metadata or {}), "user_id": self.user_id}
            meta = self._build_metadata(merged, memory_type, float(merged.get("importance", imp)), extra_metadata)
            self.collection.update(
                ids=[memory_id],
                documents=[merged["content"]],
                metadatas=[meta],
            )
        else:
            memory_id = extracted["id"]
            extra_metadata = {**(extra_metadata or {}), "user_id": self.user_id}
            meta = self._build_metadata(extracted, memory_type, imp, extra_metadata)
            self.collection.add(
                ids=[memory_id],
                documents=[extracted["content"]],
                metadatas=[meta],
            )

        self._sync_index_state(ltm)
        return memory_id

    async def ingest_async(
        self,
        ltm: "SelfBuiltLongTermMemoryImproved",
        content: str,
        **kwargs: Any,
    ) -> str:
        """异步写入：在线程池中跑同步管道，避免阻塞事件循环。"""
        return await asyncio.to_thread(self.ingest_sync, ltm, content, **kwargs)

    async def enqueue(self, item: Dict[str, Any]) -> None:
        await self._queue.put(item)

    async def _worker(self, ltm: "SelfBuiltLongTermMemoryImproved") -> None:
        while True:
            item = await self._queue.get()
            if item.get("__stop__"):
                break
            try:
                await self.ingest_async(ltm, **item)
            finally:
                self._queue.task_done()

    def start_background_worker(self, ltm: "SelfBuiltLongTermMemoryImproved") -> None:
        if self._worker_task is None or self._worker_task.done():
            self._worker_task = asyncio.create_task(self._worker(ltm))

    async def stop_background_worker(self) -> None:
        if self._worker_task:
            await self._queue.put({"__stop__": True})
            await self._worker_task

    @staticmethod
    def _build_metadata(
        extracted: Dict[str, Any],
        memory_type: str,
        importance: float,
        extra: Dict[str, Any] | None,
    ) -> Dict[str, Any]:
        return {
            "user_id": (extra or {}).get("user_id") or extracted.get("metadata", {}).get("user_id") or "",
            "memory_type": memory_type,
            "importance": importance,
            "summary": extracted["summary"],
            "key_points": json.dumps(extracted["key_points"], ensure_ascii=False),
            "timestamp": extracted.get("timestamp", datetime.now().isoformat()),
            **(extra or {}),
        }


# ---------------------------------------------------------------------------
# 长期记忆 + 检索重排 + 与短期合并拼 prompt
# ---------------------------------------------------------------------------
class SelfBuiltLongTermMemoryImproved:
    def __init__(
        self,
        user_id: str,
        *,
        embedding_model: str = "dashscope",
        persist_directory: str = "./chroma_selfbuilt_improve",
        collection_name: str = "long_term_memory_v2",
        top_k: int = 5,
        similarity_threshold: float = 0.0,
        time_decay_factor: float = 0.95,
        importance_weight: float = 0.3,
        short_term_max_turns: int = 20,
        llm=None,
    ) -> None:
        self.user_id = user_id
        self.top_k = top_k
        self.similarity_threshold = similarity_threshold
        self.time_decay_factor = time_decay_factor
        self.importance_weight = importance_weight

        self.short_term = ThreadShortTermMemory(max_turns=short_term_max_turns)

        self.client = chromadb.PersistentClient(path=persist_directory)
        self.embedding_function = self._create_embedding_function(embedding_model)
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            embedding_function=self.embedding_function,
            metadata={"user_id": user_id},
        )

        self.hybrid_retriever = HybridRetriever()
        self.documents: List[str] = []
        self.metadatas: List[Dict[str, Any]] = []
        self.ids: List[str] = []

        self._llm = llm or default_llm()
        self.compressor = MemoryCompressor(self._llm)
        self.write_pipeline = MemoryWritePipeline(
            self.collection,
            self.compressor,
            self.hybrid_retriever,
            user_id=user_id,
            on_index_refreshed=None,
        )
        self.write_pipeline._sync_index_state(self)


    def _create_embedding_function(self, model_type: str):
        kind = (model_type or "dashscope").strip().lower()

        # 阿里云百炼：OpenAI 兼容 Embeddings（推荐，无需下载 BGE）
        if kind in ("dashscope", "qwen", "aliyun", "阿里云"):
            api_key = (os.getenv("DASHSCOPE_API_KEY") or "").strip()
            if not api_key:
                raise ValueError(
                    "使用阿里云嵌入请在 .env 设置 DASHSCOPE_API_KEY（百炼控制台 API Key）"
                )
            model = os.getenv("DASHSCOPE_EMBEDDING_MODEL", "text-embedding-v3")
            base = os.getenv(
                "DASHSCOPE_EMBEDDING_BASE_URL",
                "https://dashscope.aliyuncs.com/compatible-mode/v1",
            ).rstrip("/")
            # 写入环境变量，便于 Chroma OpenAIEmbeddingFunction 内部客户端读取
            os.environ["DASHSCOPE_API_KEY"] = api_key
            return embedding_functions.OpenAIEmbeddingFunction(
                api_key=api_key,
                api_key_env_var="DASHSCOPE_API_KEY",
                model_name=model,
                api_base=base,
            )

        if kind == "bge":
            return embedding_functions.SentenceTransformerEmbeddingFunction(
                model_name=os.getenv("BGE_MODEL_NAME", "BAAI/bge-m3")
            )

        if kind == "openai":
            return embedding_functions.OpenAIEmbeddingFunction(
                model_name=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
                api_key=os.getenv("OPENAI_API_KEY"),
                api_base=(os.getenv("OPENAI_BASE_URL") or "").strip() or None,
            )

        raise ValueError(
            f"不支持的 embedding_model={model_type!r}，请用 dashscope / openai / bge"
        )

    # ----- 对外：一轮对话结束（写短期 + 异步/同步长期）-----
    def record_turn(
        self,
        thread_id: str,
        user_content: str,
        assistant_content: str | None = None,
    ) -> None:
        """模拟 Checkpoint 追加 messages；assistant 可选（仅用户侧入库长期记忆）。"""
        self.short_term.append(thread_id, "user", user_content)
        if assistant_content:
            self.short_term.append(thread_id, "assistant", assistant_content)

    def close(self) -> None:
        """释放 Chroma 客户端（Windows/Jupyter 重复运行前建议调用）。"""
        self.collection = None
        self.client = None

    def clear_all_memories(self) -> None:
        """清空本 collection 内全部记忆（不删磁盘目录，适合 Notebook 重复运行）。"""
        data = self.collection.get()
        ids = data.get("ids") or []
        if ids:
            self.collection.delete(ids=ids)
        self.documents = []
        self.metadatas = []
        self.ids = []
        self.hybrid_retriever = HybridRetriever()

    def ingest_long_term(
        self,
        content: str,
        *,
        memory_type: str = "general",
        importance: float | None = None,
        metadata: Dict[str, Any] | None = None,
    ) -> str:
        meta = {"user_id": self.user_id, **(metadata or {})}
        return self.write_pipeline.ingest_sync(
            self, content, memory_type=memory_type, importance=importance, extra_metadata=meta
        )

    async def ingest_long_term_async(self, content: str, **kwargs: Any) -> str:
        kwargs.setdefault("extra_metadata", {"user_id": self.user_id})
        return await self.write_pipeline.ingest_async(self, content, **kwargs)

    # ----- 检索 + 重排 -----
    def search_memories(
        self,
        query: str,
        *,
        memory_types: List[str] | None = None,
        top_k: int | None = None,
        use_time_decay: bool = True,
        use_importance_weight: bool = True,
    ) -> List[Dict[str, Any]]:
        k = top_k or self.top_k
        if not self.documents:
            return []

        where: Dict[str, Any] = {"user_id": self.user_id}
        if memory_types:
            where["memory_type"] = {"$in": memory_types}

        dense = self.collection.query(query_texts=[query], n_results=k * 2, where=where)
        dense_scores: List[tuple[int, float]] = []
        idx_to_vector_sim: Dict[int, float] = {}
        if dense["ids"] and dense["ids"][0]:
            for mem_id, dist in zip(dense["ids"][0], dense["distances"][0]):
                if mem_id in self.ids:
                    idx = self.ids.index(mem_id)
                    vector_sim = self._distance_to_similarity(dist)
                    idx_to_vector_sim[idx] = vector_sim
                    dense_scores.append((idx, vector_sim))

        hybrid = self.hybrid_retriever.search(query, dense_scores, k * 2)

        results: List[Dict[str, Any]] = []
        for idx, rrf_score in hybrid:
            meta = self.metadatas[idx]
            vector_sim = idx_to_vector_sim.get(idx, 0.0)
            # RRF 排名分 × 向量相似度幅度 → 随 query 变化的基础分
            score = rrf_score * vector_sim
            if use_time_decay:
                score *= self._time_decay(meta.get("timestamp", datetime.now().isoformat()))
            if use_importance_weight:
                imp = float(meta.get("importance", 0.5))
                score *= 0.5 + imp  # 乘性加权，避免常数项把分数拉平
            final_score = score
            if final_score >= self.similarity_threshold:
                results.append(
                    {
                        "id": self.ids[idx],
                        "content": self.documents[idx],
                        "summary": meta.get("summary", ""),
                        "memory_type": meta.get("memory_type", "general"),
                        "importance": meta.get("importance", 0.5),
                        "timestamp": meta.get("timestamp", ""),
                        "vector_sim": vector_sim,
                        "rrf_score": rrf_score,
                        "final_score": final_score,
                        "score": final_score,
                    }
                )
        return sorted(results, key=lambda x: x["final_score"], reverse=True)[:k]

    def _time_decay(self, timestamp: str) -> float:
        try:
            days = (datetime.now() - datetime.fromisoformat(timestamp)).days
        except ValueError:
            days = 0
        return self.time_decay_factor ** max(days, 0)

    @staticmethod
    def _distance_to_similarity(distance: float) -> float:
        return max(0.0, min(1.0, 1.0 - float(distance) / 2.0))

    # ----- 拼 prompt（短期 + 长期）-----
    def build_prompt(
        self,
        thread_id: str,
        query: str,
        *,
        long_term_hits: List[Dict[str, Any]] | None = None,
        recent_n: int = 6,
    ) -> str:
        hits = long_term_hits if long_term_hits is not None else self.search_memories(query)
        if hits:
            memory_context = "\n".join(
                f"- [{h['memory_type']}] {h['summary'] or h['content']} "
                f"(final={h['final_score']:.4f}, vector_sim={h['vector_sim']:.4f}, rrf={h['rrf_score']:.4f})"
                for h in hits
            )
        else:
            memory_context = "（未检索到相关长期记忆）"
        recent = self.short_term.format_recent(thread_id, last_n=recent_n)
        return MEMORY_PROMPT_TEMPLATE.format(
            recent_dialogue=recent,
            memory_context=memory_context,
            query=query,
        )


# ---------------------------------------------------------------------------
# 演示：短期 Checkpoint 模拟 + 写入管道 + 检索拼 prompt
# ---------------------------------------------------------------------------

print('✓ 源码 B 已执行：SelfBuiltLongTermMemoryImproved 等类已定义')

✓ 源码 B 已执行：SelfBuiltLongTermMemoryImproved 等类已定义


---
## 模块 1 · 记忆抽取

调用源码 A 里的 `MemoryCompressor.extract_memory`。


In [19]:
llm = default_llm()
compressor = MemoryCompressor(llm)

RAW = "用户喜欢住海景房，预算每晚不超过800元"
extracted = compressor.extract_memory(RAW)

print("【原文】", RAW)
print("【抽取结果】")
print(json.dumps({k: extracted[k] for k in ("summary", "key_points", "importance")}, ensure_ascii=False, indent=2))

【原文】 用户喜欢住海景房，预算每晚不超过800元
【抽取结果】
{
  "summary": "用户偏好海景房，预算每晚不超过800元",
  "key_points": [
    "喜欢海景房",
    "预算每晚不超过800元"
  ],
  "importance": 0.9
}


---
## 模块 2 · 记忆合并


In [20]:
old_memory = {
    "id": "mem-old-001",
    "content": "用户喜欢住海景房，预算每晚不超过800元",
    "summary": "用户偏好海景住宿且预算有限",
    "key_points": ["喜欢海景房", "预算800元以内", "每晚"],
    "importance": 0.8,
}
new_memory = compressor.extract_memory("用户说以后订酒店都要海景房，价格别超过800")
merged = compressor.merge_memories([old_memory, new_memory])

print("合并后:", json.dumps({k: merged[k] for k in ("summary", "key_points", "importance")}, ensure_ascii=False, indent=2))

合并后: {
  "summary": "用户今后预订酒店时必须选择海景房，且每晚价格不超过800元。",
  "key_points": [
    "偏好海景房",
    "预算上限为每晚800元",
    "此要求适用于所有未来的酒店预订"
  ],
  "importance": 0.9
}


---
## 模块 3 · 存储

调用 `ingest_long_term` 写入 Chroma。

> **Windows / 重复运行**：默认用 `get_or_reset_ltm` **复用** 同一个 `ltm`，只 `clear_all_memories()`，**不删** `chroma_notebook_demo` 文件夹（避免 WinError 32）。  
> 只有 Restart Kernel 之后，才可将下面 `FORCE_REBUILD_DIRECTORY = True` 尝试整目录重建。


In [21]:
USER_ID = "student_demo"
CHROMA_DIR = ROOT / "chroma_notebook_dem"
FORCE_REBUILD_DIRECTORY = False  # Restart Kernel 后可改 True 尝试删目录重建

ltm = get_or_reset_ltm(
    user_id=USER_ID,
    persist_directory=CHROMA_DIR,
    embedding_model="dashscope",
    collection_name="notebook_long_term_v1",
    similarity_threshold=0.0,
    force_rebuild_directory=FORCE_REBUILD_DIRECTORY,
)

samples = [
    ("用户喜欢住海景房，预算每晚不超过800元", "preference", 0.8),
    ("用户计划下个月去三亚旅游", "fact", 0.5),
    ("用户偏好海景住宿，价格控制在800以内", "preference", None),  # 近似重复 → 合并
]
for text, mtype, imp in samples:
    mid = ltm.ingest_long_term(text, memory_type=mtype, importance=imp)
    print(f"写入 id={mid[:8]}… | {mtype} | {text[:20]}…")
print(f"\n库内条数: {len(ltm.documents)}（期望 2，第3条应合并）")

写入 id=3e4bfa45… | preference | 用户喜欢住海景房，预算每晚不超过800元…
写入 id=315e6e41… | fact | 用户计划下个月去三亚旅游…
写入 id=3e4bfa45… | preference | 用户偏好海景住宿，价格控制在800以内…

库内条数: 2（期望 2，第3条应合并）


In [22]:
print("=== 库内记忆 ===")
for i, (doc_id, doc, meta) in enumerate(zip(ltm.ids, ltm.documents, ltm.metadatas), 1):
    print(f"[{i}] {meta.get('summary')} | type={meta.get('memory_type')} | imp={meta.get('importance')}")

=== 库内记忆 ===
[1] 用户喜欢海景住宿，预算每晚不超过800元。 | type=preference | imp=0.5
[2] 用户计划下个月去三亚旅游 | type=fact | imp=0.5


---
## 模块 4 · 检索

**打分链路（改进后）**：

1. **vector_sim**：Chroma 距离 → `1 - dist/2`（反映 query 与记忆的语义接近程度）
2. **rrf_score**：向量排名 + jieba BM25 排名的 RRF 融合（只看名次，不看距离幅度）
3. **base = rrf_score × vector_sim**：把排名与语义幅度结合
4. **final_score = base × time_decay × (0.5 + importance)**：时间衰减 + 重要性**乘性**加权

下面用两个 query 对比：排序仍由 RRF 主导，但 **final_score 会随 vector_sim 变化**。


In [26]:
def run_search_demo(query: str):
    print("=" * 72)
    print(f"QUERY: {query}")
    dense = ltm.collection.query(query_texts=[query], n_results=5, where={"user_id": USER_ID})
    print("\n【向量检索】")
    for mid, dist, meta in zip(dense["ids"][0], dense["distances"][0], dense["metadatas"][0]):
        sim = SelfBuiltLongTermMemoryImproved._distance_to_similarity(dist)
        print(f"  {meta.get('summary','')[:36]} | dist={dist:.4f} | vector_sim={sim:.4f}")

    hits = ltm.search_memories(query, top_k=5)
    print("\n【混合检索 search_memories — 分项 score】")
    for h in hits:
        print(
            f"  [{h['memory_type']}] {h['summary']}\n"
            f"    vector_sim={h['vector_sim']:.4f}  "
            f"rrf_score={h['rrf_score']:.4f}  "
            f"final_score={h['final_score']:.4f}"
        )
    return hits


hits_hotel = run_search_demo("你还记得我喜欢住什么样的酒店吗？")
print()
QUERY = "你还记得我喜欢住什么样的酒店吗？"
hits = hits_hotel  # 模块 5 默认用酒店 query 的结果

QUERY: 你还记得我喜欢住什么样的酒店吗？

【向量检索】
  用户喜欢海景住宿，预算每晚不超过800元。 | dist=0.4616 | vector_sim=0.7692
  用户计划下个月去三亚旅游 | dist=0.5102 | vector_sim=0.7449

【混合检索 search_memories — 分项 score】
  [preference] 用户喜欢海景住宿，预算每晚不超过800元。
    vector_sim=0.7692  rrf_score=0.0167  final_score=0.0128
  [fact] 用户计划下个月去三亚旅游
    vector_sim=0.7449  rrf_score=0.0164  final_score=0.0122



---
## 模块 5 · 使用（拼 Prompt）


In [27]:
THREAD_ID = "thread_notebook_001"
ltm.record_turn(THREAD_ID, "你好，我想规划一次旅行", "你好！请问您想去哪里旅行呢？")
prompt = ltm.build_prompt(THREAD_ID, QUERY, long_term_hits=hits)
print(prompt)

你是一个有记忆的智能助手。请结合【最近对话】与【相关长期记忆】回答用户问题。

【最近对话】（来自 Checkpoint / 本线程短期缓冲，按时间顺序）
user: 你好，我想规划一次旅行
assistant: 你好！请问您想去哪里旅行呢？
user: 你好，我想规划一次旅行
assistant: 你好！请问您想去哪里旅行呢？

【相关长期记忆】（向量检索 + 重排后的 top-k）
- [preference] 用户喜欢海景住宿，预算每晚不超过800元。 (final=0.0128, vector_sim=0.7692, rrf=0.0167)
- [fact] 用户计划下个月去三亚旅游 (final=0.0122, vector_sim=0.7449, rrf=0.0164)

【当前问题】
你还记得我喜欢住什么样的酒店吗？



In [28]:
RUN_LLM = True  # 改为 True 可让大模型根据 Prompt 回答

if RUN_LLM:
    print(llm.invoke(prompt).content)
else:
    print("RUN_LLM=False：仅展示 Prompt。改 True 可体验完整闭环。")

当然记得！根据您之前提供的信息：

- **住宿偏好**：您喜欢 **海景房**，希望能够欣赏到海边的美景。  
- **预算限制**：每晚的住宿费用 **不超过 800 元**。  

另外，您计划下个月前往 **三亚** 旅游，这正是一个拥有众多海景酒店的热门目的地。如果您需要，我可以帮您挑选几家符合这些条件的酒店，或者提供一些性价比高的海景住宿推荐。请告诉我您还有哪些具体需求（比如入住日期、是否需要早餐、是否偏好度假村或精品酒店等），我会为您进一步细化方案。


---
## 小结

1. **源码 A** 负责「读懂一句话」和「合并两条记忆」。  
2. **源码 B** 负责「存进向量库」「搜出来」「塞进 Prompt」。
4. **抽取**：非结构化对话 → 结构化记忆字段。
5. **合并**：相似记忆归并，控制库膨胀。
6. **存储**：向量库 + 元数据 + BM25 倒排索引同步刷新。
7. **检索**：语义 + 关键词混合，比单一向量更稳。
8. **使用**：短期上下文 + 长期事实一起进 Prompt，才是「有记忆的 Agent」。

